# BCICIV2a TorchEEG Training on AMD ROCm

> 基于 TorchEEG 的运动想象分类训练 — 适配 AMD Developer Cloud (ROCm)

**训练流程:**
1. 环境检测 (ROCm / CUDA / CPU)
2. 安装依赖
3. 自动下载 BCICIV2a 数据集 (如本地无)
4. 数据探索与可视化
5. 模型训练 (LOSO 交叉验证)
6. 结果展示 (曲线 / 混淆矩阵 / 报告)

**支持模型:** EEGNet / TSCeption / FBCNet / FBMSNet
**数据集:** BCI Competition IV 2a (9 subjects, 4 classes)

---
## 0. 环境检测

检测当前 AMD ROCm / CUDA 环境, 确认 GPU 可用。

In [ ]:
from utils.fixes import apply_all_fixes
apply_all_fixes()

import sys, os, torch, platform
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'OS: {platform.system()} {platform.release()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'Memory: {props.total_mem / 1024**3:.1f} GiB')
    print(f'CUDA/ROCm version: {torch.version.cuda}')
else:
    print('[WARN] No GPU detected, using CPU')

# 检查 TorchEEG
try:
    import torcheeg
    print(f'TorchEEG: {torcheeg.__version__}')
except ImportError:
    print('[WARN] TorchEEG not installed')
    print('       Run cell 1 to install')

---
## 1. 安装依赖

仅在首次运行时需要。AMD Cloud 环境通常已预装 PyTorch ROCm。

In [ ]:
!pip install -r requirements.txt -q
print('[OK] Dependencies installed')

---
## 2. 数据集准备

自动检测 `data/` 目录下是否有数据集:
- 有 → 直接加载
- 无 → 从 BNCI Horizon 2020 下载 (~700 MB) → 组装为 BCICIV2a.mat

In [ ]:
from download_data import ensure_dataset
data_dir = ensure_dataset(download_if_missing=True, assemble=True)
print(f'Data directory: {data_dir}')

---
## 3. 数据探索

查看数据集基本信息: 形状、标签分布、受试者信息。

In [ ]:
import scipy.signal
from scipy.signal.windows import hann
scipy.signal.hann = hann

import numpy as np
from utils.data_utils import load_data

data, labels, subjects, runs = load_data(data_dir)
print(f'Data shape:       {data.shape}')
print(f'Labels shape:     {labels.shape}')
print(f'Subjects:         {np.unique(subjects)}')
print(f'Label distribution: {np.bincount(labels.flatten().astype(int))}')
print(f'Per subject: {[int((subjects==s).sum()) for s in np.unique(subjects)]}')

In [ ]:
import matplotlib.pyplot as plt

# 可视化一个 trial 的 EEG 信号
idx = 0
plt.figure(figsize=(14, 6))
plt.imshow(data[idx], aspect='auto', cmap='RdBu_r',
           extent=[0, 800/250, 22, 1])
plt.colorbar(label='\u03bcV')
plt.xlabel('Time (s)')
plt.ylabel('Channel')
plt.title(f'Trial {idx}, Subject {subjects[idx]}, Class {labels[idx]}')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 数据预处理（只需跑一次） ──
# 把 BandSignal FFT 等重计算永久存到磁盘，训练加 --use-preprocessed 直接加载
!python preprocess_dataset.py --models EEGNet TSCeption FBCNet FBMSNet

print('[OK] Preprocessing done. Now you can add --use-preprocessed to training.')

---
## 4. 模型训练

使用 Leave-One-Subject-Out (LOSO) 交叉验证训练选定模型。

支持以下参数:
- `--models`: EEGNet, TSCeption, FBCNet, FBMSNet
- `--epochs`: 训练轮数 (默认 50, 建议 30)
- `--batch-size`: 批次大小 (AMD 16GiB GPU 可用 128)
- `--lr`: 学习率
- `--use-augmentation`: 启用数据增强
- `--use-preprocessed`: 使用预处理 .pt 文件（需先运行预处理代码格，跳过 BandSignal FFT）

In [ ]:
# ====================================================
# 训练配置 — 在这里修改参数
# ====================================================
MODELS = ['EEGNet', 'TSCeption', 'FBCNet', 'FBMSNet']   # 可选模型列表
EPOCHS = 30                                  # 训练轮数
BATCH_SIZE = 64                              # 批次大小
LR = 0.001                                   # 学习率
USE_AUGMENTATION = False                     # 数据增强
# 自动检测预处理数据: 如果 data/preprocessed/ 下有 .pt 文件则自动启用
import os as _os
USE_PREPROCESSED = _os.path.exists('data/preprocessed/FBCNet_data.pt')
if USE_PREPROCESSED:
    print('[AUTO] 检测到预处理 .pt 文件，启用 use_preprocessed')
RESULTS_DIR = 'results'                      # 输出目录

print(f'Models: {MODELS}')
print(f'Epochs: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Learning rate: {LR}')
print(f'Augmentation: {USE_AUGMENTATION}')
print(f'Preprocessed: {USE_PREPROCESSED}')

In [ ]:
import time, json, os
from datetime import datetime
from train import run_loso_experiment
from utils.model_utils import get_device, verify_device
from utils.vis_utils import plot_model_comparison, generate_report

device = get_device()
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Using device: {device}')
print(f'Starting LOSO training for {len(MODELS)} models...\n')

all_results = {}
run_dirs = []
for model_name in MODELS:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = os.path.join(RESULTS_DIR, f'{model_name}_{timestamp}')
    os.makedirs(run_dir, exist_ok=True)

    train_cfg = {
        'device': device,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'weight_decay': 0.001,
        'chunk_size': 800,
        'results_dir': RESULTS_DIR,
        'run_dir': run_dir,
        'data_dir': 'data',
        'use_augmentation': USE_AUGMENTATION,
        'use_preprocessed': USE_PREPROCESSED,
    }

    print(f'\n{"="*60}')
    print(f'TRAINING: {model_name}')
    print(f'  Results: {run_dir}')
    print(f'{"="*60}')

    t0 = time.time()
    result = run_loso_experiment(model_name, data, labels, subjects, train_cfg)
    elapsed = time.time() - t0
    all_results[model_name] = result
    run_dirs.append(run_dir)

    print(f'\n[{model_name}] Mean: {result["mean_accuracy"]:.4f} '
          f'± {result["std_accuracy"]:.4f}, Time: {elapsed:.1f}s')

# 生成对比图和报告
print('\n[INFO] Generating comparison & report...')
plot_model_comparison(all_results, RESULTS_DIR)
generate_report(all_results, RESULTS_DIR, {
    'device': device,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'models': ', '.join(MODELS),
    'dataset': 'BCICIV2a (9 subjects, 4 classes)',
    'run_dirs': ', '.join(run_dirs),
})
print(f'[OK] Results saved to {RESULTS_DIR}/')
print(f'[OK] Run directories:')
for d in run_dirs:
    print(f'  {d}/')

---
## 5. 训练结果展示

查看训练生成的图表和报告。

In [ ]:
from IPython.display import Image, display

# 模型对比图
comp_path = os.path.join(RESULTS_DIR, 'model_comparison.png')
if os.path.exists(comp_path):
    display(Image(filename=comp_path))
else:
    print('Model comparison not yet generated. Run training first.')

# 各模型结果目录
if run_dirs:
    print('\\nResult directories:')
    for d in run_dirs:
        print(f'  📁 {d}/')

In [ ]:
# 各模型各受试者结果表
print('=' * 100)
print('PER-SUBJECT RESULTS')
print('=' * 100)

header = f'{"Model":<12} {"S1":<8} {"S2":<8} {"S3":<8} {"S4":<8} {"S5":<8} {"S6":<8} {"S7":<8} {"S8":<8} {"S9":<8} {"Mean":<8}'
print(header)
print('-' * len(header))

for m_name in sorted(all_results.keys(),
                     key=lambda m: all_results[m]['mean_accuracy'], reverse=True):
    r = all_results[m_name]
    accs = [sr['accuracy'] for sr in r['per_subject']]
    row = f'{m_name:<12}'
    for a in accs:
        row += f'{a:.4f}  '
    row += f'{r["mean_accuracy"]:.4f}'
    print(row)

print()
print('Run directories:')
for d in run_dirs:
    m_name = os.path.basename(d).rsplit('_', 1)[0]
    print(f'  {m_name}: {d}/')

In [ ]:
# 展示各模型第一个受试者的训练曲线
if run_dirs:
    for d in run_dirs:
        m_name = os.path.basename(d).rsplit('_', 1)[0]
        curve_path = os.path.join(d, 'S01', 'curves.png')
        if os.path.exists(curve_path):
            print(f'{m_name} S01:')
            display(Image(filename=curve_path))
        else:
            print(f'{m_name}: Training curves not yet generated.')
else:
    print('No training results found. Run training first.')

In [ ]:
# 混淆矩阵（各模型第一个受试者）
if run_dirs:
    for d in run_dirs:
        m_name = os.path.basename(d).rsplit('_', 1)[0]
        cm_path = os.path.join(d, 'S01', 'cm.png')
        if os.path.exists(cm_path):
            print(f'{m_name} S01:')
            display(Image(filename=cm_path))
        else:
            print(f'{m_name}: Confusion matrix not yet generated.')
else:
    print('No training results found. Run training first.')

---
## 6. 训练报告

完整的 Markdown 训练报告包含:
- 实验配置
- 模型对比表格
- 各模型 9 受试者详细结果
- 混淆矩阵和训练曲线引用
- 结论与优化建议

In [ ]:
from IPython.display import Markdown

report_path = os.path.join(RESULTS_DIR, 'TRAINING_REPORT.md')
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        md = f.read()
    display(Markdown(md[:3000] + '...'))  # 预览前3000字符
    print(f'\nFull report: {report_path}')
else:
    print('Report not yet generated.')

---
## 7. 实验总结

| 指标 | 说明 |
|------|------|
| 数据集 | BCICIV2a (9 subjects, 4 classes, 22 channels, 250Hz) |
| 方法 | Leave-One-Subject-Out Cross Validation |
| 模型 | EEGNet / TSCeption / FBCNet / FBMSNet |
| 评价指标 | Accuracy / Kappa / F1 / Precision / Recall / AUC |

**最佳模型:** 
- 训练完成后在上方结果表中查看最高准确率的模型

**后续优化方向:**
- 频带优化: 针对不同受试者选择最优频带子集
- 时间窗口: 探索 cue 后 0.5-2.5s 窗口
- 数据增强: 噪声 / 裁剪 / 频带扰动
- 模型集成: 多个模型预测融合
- 迁移学习: 利用源域受试者数据提升目标受试者性能

---
## 附录: TorchEEG Transforms 测试

对比不同 Transforms pipeline 的效果。

In [ ]:
"""
# 附录: Transforms 测试 (可选)
# 取消注释以运行:
#
# import scipy.signal
# from scipy.signal.windows import hann
# scipy.signal.hann = hann
#
# from test_torcheeg_features import test_03_transforms
# transform_results = test_03_transforms(data, labels, subjects)
# from IPython.display import Image
# tp = 'results/feature_tests/transforms_comparison.png'
# if os.path.exists(tp):
#     display(Image(filename=tp))
"""